In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.manual_seed(42)
np.random.seed(42)


Device: cuda


In [ ]:
x_min, x_max = 0.0, 2*np.pi
T_final = 0.3

def true_solution(x, t):
    return torch.exp(-t)*torch.sin(x) + \
           torch.exp(-81*t)*torch.sin(3*x)


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=128, num_layers=4):
        super().__init__()

        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.SiLU())

        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.SiLU())

        layers.append(nn.Linear(hidden_dim, output_dim))

        self.model = nn.Sequential(*layers)
        self.init_weights()

    def init_weights(self):
        for m in self.model:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.model(x)


In [ ]:
def compute_l2_error_physical(model):
    x = torch.linspace(x_min, x_max, 512).to(device)
    t_vals = torch.tensor([0.05,0.1,0.2]).to(device)

    X,T = torch.meshgrid(x,t_vals,indexing="ij")
    X_flat = X.reshape(-1,1)
    T_flat = T.reshape(-1,1)

    with torch.no_grad():
        u_pred = model(torch.cat([X_flat,T_flat],dim=1))
        u_true = true_solution(X_flat,T_flat)

    return torch.norm(u_pred-u_true)/torch.norm(u_true)


In [ ]:
# Training data
N_r = 6000
N_ic = 512

x_r = torch.rand(N_r,1).to(device)*(x_max-x_min) + x_min
t_r = torch.rand(N_r,1).to(device)*T_final

x_ic = torch.rand(N_ic,1).to(device)*(x_max-x_min) + x_min
t_ic = torch.zeros_like(x_ic).to(device)

u_ic_true = true_solution(x_ic, t_ic).detach()


In [ ]:
def pinn_residual(model, x, t):
    x.requires_grad_(True)
    t.requires_grad_(True)

    u = model(torch.cat([x,t],dim=1))

    # time derivative
    u_t = torch.autograd.grad(
        u, t,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    # first spatial derivative
    u_x = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    # second
    u_xx = torch.autograd.grad(
        u_x, x,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True
    )[0]

    # third
    u_xxx = torch.autograd.grad(
        u_xx, x,
        grad_outputs=torch.ones_like(u_xx),
        create_graph=True
    )[0]

    # fourth
    u_xxxx = torch.autograd.grad(
        u_xxx, x,
        grad_outputs=torch.ones_like(u_xxx),
        create_graph=True
    )[0]

    return u_t + u_xxxx


In [ ]:
pinn = MLP(2,1).to(device)
optimizer = torch.optim.Adam(pinn.parameters(), lr=1e-3)

epochs = 4000

history_pinn = {
    "loss": [],
    "l2": [],
    "grad_norm": [],
    "epoch_time": []
}

torch.cuda.reset_peak_memory_stats()
start_total = time.time()

for epoch in range(epochs):

    start_epoch = time.time()
    optimizer.zero_grad()

    # PDE residual
    res = pinn_residual(pinn, x_r, t_r)
    loss_pde = torch.mean(res**2)

    # IC loss
    u_ic_pred = pinn(torch.cat([x_ic,t_ic],dim=1))
    loss_ic = torch.mean((u_ic_pred - u_ic_true)**2)

    loss = loss_pde + loss_ic

    loss.backward()

    # gradient norm
    total_norm = 0
    for p in pinn.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item()**2
    total_norm = total_norm**0.5

    optimizer.step()

    epoch_time = time.time() - start_epoch
    l2 = compute_l2_error_physical(pinn)

    history_pinn["loss"].append(loss.item())
    history_pinn["l2"].append(l2.item())
    history_pinn["grad_norm"].append(total_norm)
    history_pinn["epoch_time"].append(epoch_time)

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss {loss.item():.3e}, L2 {l2.item():.3e}, GradNorm {total_norm:.3e}")


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 0, Loss 9.198e-01, L2 9.120e-01, GradNorm 6.703e-01
Epoch 200, Loss 4.065e-01, L2 4.357e-01, GradNorm 1.284e-03
Epoch 400, Loss 4.060e-01, L2 4.341e-01, GradNorm 1.097e-03
Epoch 600, Loss 4.054e-01, L2 4.363e-01, GradNorm 1.006e-02
Epoch 800, Loss 4.047e-01, L2 4.315e-01, GradNorm 8.670e-02
Epoch 1000, Loss 4.027e-01, L2 4.472e-01, GradNorm 7.934e-03
Epoch 1200, Loss 3.993e-01, L2 4.739e-01, GradNorm 6.124e-02
Epoch 1400, Loss 4.287e-01, L2 5.379e-01, GradNorm 2.255e+00
Epoch 1600, Loss 3.897e-01, L2 5.458e-01, GradNorm 3.569e-02
Epoch 1800, Loss 3.769e-01, L2 6.043e-01, GradNorm 8.304e-02
Epoch 2000, Loss 3.681e-01, L2 6.211e-01, GradNorm 1.895e+00
Epoch 2200, Loss 3.172e-01, L2 6.936e-01, GradNorm 2.170e+00
Epoch 2400, Loss 2.644e-01, L2 6.784e-01, GradNorm 7.144e+00
Epoch 2600, Loss 1.578e-01, L2 6.691e-01, GradNorm 5.009e+00
Epoch 2800, Loss 1.571e-01, L2 7.055e-01, GradNorm 1.851e+01
Epoch 3000, Loss 7.683e-02, L2 7.480e-01, GradNorm 1.002e+01
Epoch 3200, Loss 1.019e-01, L2 

In [ ]:
total_time_pinn = time.time() - start_total
memory_pinn = torch.cuda.max_memory_allocated()/(1024**2)

print("PINN Total Time:", total_time_pinn)
print("PINN Peak Memory (MB):", memory_pinn)


PINN Total Time: 461.53640270233154
PINN Peak Memory (MB): 1115.16015625


In [ ]:
K = 8
k_vals = torch.arange(-K, K+1).float().to(device)
num_modes = len(k_vals)

N_t = 1000
t_train = torch.rand(N_t,1).to(device) * T_final


In [ ]:
sinn = MLP(2,2).to(device)
optimizer_sinn = torch.optim.Adam(sinn.parameters(), lr=1e-3)


In [ ]:
def sinn_residual(model, k, t):
    t.requires_grad_(True)

    inputs = torch.cat([k,t],dim=1)
    u_hat = model(inputs)

    u_real = u_hat[:,0:1]
    u_imag = u_hat[:,1:2]

    u_real_t = torch.autograd.grad(
        u_real, t,
        grad_outputs=torch.ones_like(u_real),
        create_graph=True
    )[0]

    u_imag_t = torch.autograd.grad(
        u_imag, t,
        grad_outputs=torch.ones_like(u_imag),
        create_graph=True
    )[0]

    # TRUE operator (no normalization here)
    k4 = k**4

    res_real = u_real_t + k4 * u_real
    res_imag = u_imag_t + k4 * u_imag

    return res_real, res_imag


In [ ]:
def true_fourier_ic(k_vals):
    real = torch.zeros_like(k_vals)
    imag = torch.zeros_like(k_vals)

    imag[k_vals==1] = -0.5
    imag[k_vals==-1] = 0.5
    imag[k_vals==3] = -0.5
    imag[k_vals==-3] = 0.5

    return torch.stack([real, imag], dim=1)


In [ ]:
def compute_l2_error_sinn(model):
    x = torch.linspace(x_min, x_max, 512).to(device)
    t_vals = torch.tensor([0.05,0.1,0.2]).to(device)

    X,T = torch.meshgrid(x,t_vals,indexing="ij")
    X_flat = X.reshape(-1,1)
    T_flat = T.reshape(-1,1)

    u_pred = torch.zeros_like(X_flat)

    with torch.no_grad():
        for k in k_vals:
            k_tensor = torch.ones_like(T_flat) * k
            coeff = model(torch.cat([k_tensor,T_flat],dim=1))
            real = coeff[:,0:1]
            imag = coeff[:,1:2]

            u_pred += real*torch.cos(k*X_flat) - \
                      imag*torch.sin(k*X_flat)

    u_true = true_solution(X_flat,T_flat)

    return torch.norm(u_pred-u_true)/torch.norm(u_true)


In [ ]:
epochs = 4000

history_sinn = {
    "loss": [],
    "l2": [],
    "grad_norm": [],
    "epoch_time": []
}

torch.cuda.reset_peak_memory_stats()
start_total = time.time()

for epoch in range(epochs):

    start_epoch = time.time()
    optimizer_sinn.zero_grad()

    res_real, res_imag = sinn_residual(sinn, k_repeat, t_repeat)

    loss_pde = torch.mean(res_real**2) + torch.mean(res_imag**2)

    # IC
    k_ic = k_vals.reshape(-1,1)
    t_ic_spec = torch.zeros_like(k_ic)

    pred_ic = sinn(torch.cat([k_ic,t_ic_spec],dim=1))
    true_ic = true_fourier_ic(k_vals).to(device)

    loss_ic = torch.mean((pred_ic - true_ic)**2)

    loss = loss_pde + loss_ic

    loss.backward()

    # gradient norm
    total_norm = 0
    for p in sinn.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item()**2
    total_norm = total_norm**0.5

    optimizer_sinn.step()

    epoch_time = time.time() - start_epoch

    l2 = compute_l2_error_sinn(sinn)

    history_sinn["loss"].append(loss.item())
    history_sinn["l2"].append(l2.item())
    history_sinn["grad_norm"].append(total_norm)
    history_sinn["epoch_time"].append(epoch_time)

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss {loss.item():.3e}, L2 {l2.item():.3e}, GradNorm {total_norm:.3e}")


Epoch 0, Loss 3.713e+04, L2 1.076e+00, GradNorm 1.277e+06
Epoch 200, Loss 6.719e-01, L2 9.994e-01, GradNorm 5.755e+01
Epoch 400, Loss 2.145e-01, L2 9.993e-01, GradNorm 1.276e+01
Epoch 600, Loss 1.515e-01, L2 9.993e-01, GradNorm 8.212e+00
Epoch 800, Loss 1.080e-01, L2 9.993e-01, GradNorm 5.719e+02
Epoch 1000, Loss 8.016e-02, L2 9.993e-01, GradNorm 3.794e+00
Epoch 1200, Loss 7.078e-02, L2 9.994e-01, GradNorm 2.835e+00
Epoch 1400, Loss 5.229e-02, L2 9.994e-01, GradNorm 4.318e+01
Epoch 1600, Loss 5.025e-02, L2 9.994e-01, GradNorm 1.242e+00
Epoch 1800, Loss 4.873e-02, L2 9.994e-01, GradNorm 9.871e-01
Epoch 2000, Loss 4.610e-02, L2 9.995e-01, GradNorm 1.226e+01
Epoch 2200, Loss 4.432e-02, L2 9.995e-01, GradNorm 7.821e-01
Epoch 2400, Loss 4.335e-02, L2 9.995e-01, GradNorm 7.496e-01
Epoch 2600, Loss 4.204e-02, L2 9.995e-01, GradNorm 4.124e+01
Epoch 2800, Loss 4.098e-02, L2 9.995e-01, GradNorm 6.396e-01
Epoch 3000, Loss 4.080e-02, L2 9.995e-01, GradNorm 7.129e+01
Epoch 3200, Loss 3.803e-02, L2 

In [ ]:
total_time_sinn = time.time() - start_total
memory_sinn = torch.cuda.max_memory_allocated()/(1024**2)

print("SINN Total Time:", total_time_sinn)
print("SINN Peak Memory (MB):", memory_sinn)


SINN Total Time: 113.51711082458496
SINN Peak Memory (MB): 501.91455078125
